# Qwen3.5-0.8B Full Dataset Training on Colab T4

Train Qwen3.5-0.8B on full DialogSum dataset (12,460 samples) using LoRA.

**Usage:** Run cells 1→2→3→4→5→6 in order. That's it.

**Experiments:**
- exp0_qwen: Baseline (no length control)
- exp1_qwen: Length-controllable summarization  
- exp1_multi_qwen: Multi-task (summarization + topic generation)

In [ ]:
# @title Cell 1: Install dependencies
import subprocess, sys

# Upgrade transformers FIRST (required for Qwen3.5)
!pip install -q --upgrade transformers
!pip install -q datasets peft rouge-score tqdm accelerate

# Try flash-attn (optional, speeds up training ~2x)
try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "flash-attn", "--no-build-isolation"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    USE_FLASH_ATTN = True
    print("Flash Attention 2: installed")
except Exception:
    USE_FLASH_ATTN = False
    print("Flash Attention 2: not available, using SDPA (slightly slower but fine)")

print("All dependencies installed.")
print("IMPORTANT: If Colab prompts to restart the runtime, click RESTART, then skip this cell and run Cell 2.")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# @title Cell 2: Initialize everything (run after Cell 1 or after runtime restart)
import json, random, numpy as np
from pathlib import Path
from typing import Dict, List

import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    Trainer, TrainingArguments,
    __version__ as transformers_version,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftConfig, PeftModel
from datasets import load_dataset
from tqdm import tqdm

# ── Globals ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Transformers: {transformers_version}")
print(f"Device: {device}", end="")
if torch.cuda.is_available():
    print(f" ({torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print()
print(f"Flash Attention: {USE_FLASH_ATTN}")

# ── Tokenizer ────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B", padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer ready: {len(tokenizer)} tokens")

# ── Dataset ──────────────────────────────────────────
print("Loading DialogSum dataset...")
ds = load_dataset("knkarthick/dialogsum")
print(f"Train: {len(ds['train'])}, Val: {len(ds['validation'])}, Test: {len(ds['test'])}")

# ── Constants ────────────────────────────────────────
SHORT_MAX = 15
MEDIUM_MAX = 35
BUCKET_RANGES = {"SHORT": (5, 15), "MEDIUM": (16, 35), "LONG": (36, 999)}
LENGTH_INSTRUCTIONS = {
    "SHORT": "Write a very brief one-sentence summary of the dialogue in 5 to 15 words.",
    "MEDIUM": "Write a short summary of the dialogue in 16 to 35 words.",
    "LONG": "Write a detailed summary of the dialogue in more than 35 words.",
}

def get_length_bucket(summary: str) -> str:
    wc = len(summary.split())
    if wc <= SHORT_MAX: return "SHORT"
    if wc <= MEDIUM_MAX: return "MEDIUM"
    return "LONG"

print("\nInitialization complete. Ready to train.")

In [ ]:
# @title Cell 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Cell 4: Training + Evaluation functions

MODEL_NAME = "Qwen/Qwen3.5-0.8B"
OUTPUT_BASE = Path("/content/drive/MyDrive/qwen_colab")
MAX_INPUT = 256
MAX_TARGET = 64

# ── Data preprocessing ────────────────────────────────
def build_training_pairs(dialogue, summary, topic, use_length_tokens, multitask):
    """Build (prompt, target) pairs for one dialogue."""
    pairs = []
    # Summarize task
    if use_length_tokens:
        bucket = get_length_bucket(summary)
        instr = LENGTH_INSTRUCTIONS[bucket]
        prompt = f"Summarize the following dialogue.\nInstruction: {instr}\nDialogue:\n{dialogue}\nSummary: "
    else:
        prompt = f"Summarize the following dialogue.\nDialogue:\n{dialogue}\nSummary: "
    pairs.append((prompt, summary))
    # Topic task
    if multitask:
        topic_prompt = f"What is the topic of the following dialogue? Answer in a short phrase.\nDialogue:\n{dialogue}\nTopic: "
        pairs.append((topic_prompt, topic))
    return pairs

def preprocess_batch(batch, tok, use_length_tokens=False, multitask=False):
    """Tokenize a batch of dialogues for causal LM training."""
    all_input_ids, all_attention_mask, all_labels = [], [], []
    max_total = MAX_INPUT + MAX_TARGET
    for dialogue, summary, topic in zip(batch["dialogue"], batch["summary"], batch["topic"]):
        for prompt_text, target_text in build_training_pairs(dialogue, summary, topic, use_length_tokens, multitask):
            prompt_ids = tok.encode(prompt_text, add_special_tokens=False)
            target_ids = tok.encode(target_text + tok.eos_token, add_special_tokens=False)
            if len(prompt_ids) + len(target_ids) > max_total:
                prompt_ids = prompt_ids[-(max_total - len(target_ids)):]
            full_ids = prompt_ids + target_ids
            all_input_ids.append(full_ids)
            all_attention_mask.append([1] * len(full_ids))
            all_labels.append([-100] * len(prompt_ids) + list(target_ids))
    return {"input_ids": all_input_ids, "attention_mask": all_attention_mask, "labels": all_labels}

# ── Model loading ─────────────────────────────────────
def load_model():
    attn = "flash_attention_2" if USE_FLASH_ATTN else "sdpa"
    print(f"  Loading {MODEL_NAME} (attn={attn})...")
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, attn_implementation=attn)
    lora_cfg = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model

# ── Data collator ─────────────────────────────────────
class CausalLMCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id
    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad = max_len - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [self.pad_token_id] * pad)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad)
            batch["labels"].append(f["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

# ── Training ──────────────────────────────────────────
def train_experiment(exp_name, use_length_tokens=False, multitask=False, epochs=5, batch_size=4):
    print(f"\n{'='*60}")
    print(f"  {exp_name}  |  length={use_length_tokens}  multitask={multitask}  bs={batch_size}  epochs={epochs}")
    print(f"{'='*60}")

    # Preprocess
    print("Preprocessing...")
    train_tok = ds["train"].map(
        preprocess_batch, fn_kwargs={"tok": tokenizer, "use_length_tokens": use_length_tokens, "multitask": multitask},
        batched=True, batch_size=1000, remove_columns=ds["train"].column_names, desc="Train",
    )
    val_tok = ds["validation"].map(
        preprocess_batch, fn_kwargs={"tok": tokenizer, "use_length_tokens": use_length_tokens, "multitask": multitask},
        batched=True, batch_size=500, remove_columns=ds["validation"].column_names, desc="Val",
    )
    print(f"  train={len(train_tok)}  val={len(val_tok)}")

    # Model
    model = load_model()

    # Output dir
    output_dir = OUTPUT_BASE / exp_name
    output_dir.mkdir(parents=True, exist_ok=True)

    # Training
    args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=5e-5,
        warmup_steps=100,
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        bf16=True,
        gradient_checkpointing=True,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=CausalLMCollator(tokenizer.pad_token_id),
    )

    print("Training...")
    result = trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)

    # Save config
    cfg = {"experiment": exp_name, "model": MODEL_NAME, "use_length_tokens": use_length_tokens,
           "multitask": multitask, "train_samples": len(train_tok), "epochs": epochs, "batch_size": batch_size}
    with open(output_dir / "config.json", "w") as f:
        json.dump(cfg, f, indent=2)

    loss = result.metrics.get("train_loss", float("nan"))
    print(f"Done. Loss={loss:.4f}  Saved to {output_dir}")
    return trainer

# ── Evaluation ────────────────────────────────────────
def evaluate_experiment(exp_name, max_samples=500):
    model_dir = OUTPUT_BASE / exp_name
    if not model_dir.exists():
        print(f"SKIP {exp_name}: not found at {model_dir}")
        return None

    with open(model_dir / "config.json") as f:
        cfg = json.load(f)
    use_len = cfg.get("use_length_tokens", False)

    # Load model
    attn = "flash_attention_2" if USE_FLASH_ATTN else "sdpa"
    peft_config = PeftConfig.from_pretrained(str(model_dir))
    base = AutoModelForCausalLM.from_pretrained(peft_config.base_model_name_or_path, dtype=torch.bfloat16, attn_implementation=attn)
    base.resize_token_embeddings(len(tokenizer))
    model = PeftModel.from_pretrained(base, str(model_dir)).to(device).eval()

    # Build eval prompts
    test_data = ds["test"].select(range(min(max_samples, len(ds["test"]))))
    prompts, refs, buckets = [], [], []
    for row in test_data:
        refs.append(row["summary"])
        buckets.append(get_length_bucket(row["summary"]))
        if use_len:
            b = get_length_bucket(row["summary"])
            instr = LENGTH_INSTRUCTIONS[b]
            prompts.append(f"Summarize the following dialogue.\nInstruction: {instr}\nDialogue:\n{row['dialogue']}\nSummary: ")
        else:
            prompts.append(f"Summarize the following dialogue.\nDialogue:\n{row['dialogue']}\nSummary: ")

    # Generate
    print(f"Generating {len(prompts)} summaries for {exp_name}...")
    preds = []
    for i in tqdm(range(0, len(prompts), 8)):
        enc = tokenizer(prompts[i:i+8], max_length=MAX_INPUT, truncation=True, padding=True, return_tensors="pt").to(device)
        plen = enc["input_ids"].shape[1]
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_TARGET, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id, do_sample=False)
        preds.extend(d.strip() for d in tokenizer.batch_decode(out[:, plen:], skip_special_tokens=True))

    # ROUGE
    from rouge_score import rouge_scorer as rs
    scorer = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rL = [], [], []
    for pred, ref in zip(preds, refs):
        s = scorer.score(ref, pred)
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rL.append(s["rougeL"].fmeasure)

    results = {
        "experiment": exp_name, "n": len(preds),
        "rouge1": round(sum(r1)/len(r1)*100, 2),
        "rouge2": round(sum(r2)/len(r2)*100, 2),
        "rougeL": round(sum(rL)/len(rL)*100, 2),
    }

    # Length accuracy
    if use_len:
        correct = sum(1 for p, b in zip(preds, buckets) if BUCKET_RANGES[b][0] <= len(p.split()) <= BUCKET_RANGES[b][1])
        results["length_acc"] = round(correct / len(preds), 4)
        for bname in ["SHORT", "MEDIUM", "LONG"]:
            idxs = [i for i, b in enumerate(buckets) if b == bname]
            if idxs:
                bc = sum(1 for i in idxs if BUCKET_RANGES[buckets[i]][0] <= len(preds[i].split()) <= BUCKET_RANGES[buckets[i]][1])
                results[f"len_acc_{bname.lower()}"] = round(bc / len(idxs), 4)

    # Save & print
    with open(model_dir / "eval_results.json", "w") as f:
        json.dump(results, f, indent=2)

    print(f"\n{'='*50}")
    print(f"{exp_name}  (n={results['n']})")
    print(f"{'='*50}")
    print(f"  ROUGE-1: {results['rouge1']:.2f}")
    print(f"  ROUGE-2: {results['rouge2']:.2f}")
    print(f"  ROUGE-L: {results['rougeL']:.2f}")
    if "length_acc" in results:
        print(f"  Length Acc: {results['length_acc']*100:.1f}%")
        for bname in ["SHORT", "MEDIUM", "LONG"]:
            k = f"len_acc_{bname.lower()}"
            if k in results:
                print(f"    {bname}: {results[k]*100:.1f}%")
    return results

print("Functions defined. Ready to train.")

## Run Experiments

Run cells 5, 6, 7 sequentially. Each takes ~45-90 min on T4.

If OOM, change `batch_size=16` to `batch_size=8`.

In [ ]:
# @title Cell 5: Exp0 — Baseline (no length control)
train_experiment("exp0_qwen_full", use_length_tokens=False, multitask=False, epochs=5, batch_size=4)
evaluate_experiment("exp0_qwen_full")

In [ ]:
# @title Cell 6: Exp1 — Length-controllable summarization
train_experiment("exp1_qwen_full", use_length_tokens=True, multitask=False, epochs=5, batch_size=4)
evaluate_experiment("exp1_qwen_full")

In [ ]:
# @title Cell 7: Exp2 — Multi-task (summarization + topic)
train_experiment("exp1_multi_qwen_full", use_length_tokens=True, multitask=True, epochs=5, batch_size=4)
evaluate_experiment("exp1_multi_qwen_full")

In [ ]:
# @title Cell 8: Summary table
print("\n" + "="*70)
print("ALL RESULTS")
print("="*70)
for exp in ["exp0_qwen_full", "exp1_qwen_full", "exp1_multi_qwen_full"]:
    p = OUTPUT_BASE / exp / "eval_results.json"
    if p.exists():
        with open(p) as f:
            r = json.load(f)
        print(f"\n{exp}:")
        print(f"  ROUGE-1={r['rouge1']:.2f}  ROUGE-2={r['rouge2']:.2f}  ROUGE-L={r['rougeL']:.2f}", end="")
        if "length_acc" in r:
            print(f"  LenAcc={r['length_acc']*100:.1f}%", end="")
        print()
    else:
        print(f"\n{exp}: not found")

In [ ]:
# @title Cell 9: Download all checkpoints
!cd /content/drive/MyDrive/qwen_colab && zip -r qwen_models.zip exp0_qwen_full exp1_qwen_full exp1_multi_qwen_full
from google.colab import files
files.download('/content/drive/MyDrive/qwen_colab/qwen_models.zip')